In [1]:
# ==========================================================
# NLP Assignment 4: Fine-Tuning BERT on IMDB Dataset
# Author: Bhavana S
# Description: Text Classification using BERT (Hugging Face)
# ==========================================================


# ===============================
# STEP 1: INSTALL LIBRARIES
# ===============================
!pip install transformers datasets scikit-learn torch matplotlib seaborn


# ===============================
# STEP 2: IMPORT LIBRARIES
# ===============================
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


# ===============================
# STEP 3: LOAD DATASET (IMDB)
# ===============================
print("Loading dataset...")
dataset = load_dataset("imdb")

# Convert to pandas (for preprocessing)
train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

print(train_df.head())


# ===============================
# STEP 4: DATA PREPROCESSING
# ===============================
def preprocess_text(text):
    """
    Basic text cleaning
    """
    text = text.lower()
    text = text.replace("<br />", "")
    return text

train_df['text'] = train_df['text'].apply(preprocess_text)
test_df['text'] = test_df['text'].apply(preprocess_text)


# ===============================
# STEP 5: SPLIT DATA (TRAIN/VAL)
# ===============================
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'].tolist(),
    train_df['label'].tolist(),
    test_size=0.1
)


# ===============================
# STEP 6: TOKENIZATION
# ===============================
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)
test_encodings = tokenizer(test_df['text'].tolist(), truncation=True, padding=True)


# ===============================
# STEP 7: CREATE DATASET CLASS
# ===============================
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDbDataset(train_encodings, train_labels)
val_dataset = IMDbDataset(val_encodings, val_labels)
test_dataset = IMDbDataset(test_encodings, test_df['label'].tolist())


# ===============================
# STEP 8: LOAD MODEL
# ===============================
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


# ===============================
# STEP 9: METRICS FUNCTION
# ===============================
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


# ===============================
# STEP 10: TRAINING ARGUMENTS
# ===============================
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    logging_dir='./logs',
    save_strategy="epoch"
)


# ===============================
# STEP 11: TRAIN MODEL
# ===============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Training started...")
trainer.train()


# ===============================
# STEP 12: EVALUATION
# ===============================
print("Evaluating model...")
results = trainer.evaluate()
print(results)


# ===============================
# STEP 13: CONFUSION MATRIX
# ===============================
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

cm = confusion_matrix(labels, preds)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


# ===============================
# STEP 14: EXPERIMENT 1 (FREEZE BERT)
# ===============================
for param in model.bert.parameters():
    param.requires_grad = False

print("BERT layers frozen. Train only classifier layer.")


# ===============================
# STEP 15: EXPERIMENT 2 (LAST 2 LAYERS TRAIN)
# ===============================
for param in model.bert.encoder.layer[-2:].parameters():
    param.requires_grad = True

print("Last 2 BERT layers fine-tuned.")

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'